# Examensarbete: Svenska filmer per årtionde

Programmet hämtar filmuppgifter från Wikidata och kompletterar dem med IMDb-betyg och antal röster. Resultatet sparas lokalt och kan visas som filmlista, topplista eller diagram.

Notebooken visar programmets delar steg för steg. Kör cellerna uppifrån och ned. En funktion som definieras med `def` körs först när den anropas. Därför skriver vissa celler bara att funktionerna är laddade, medan andra innehåller demonstrationer längst ned.

För demonstrationerna används `data/1990.json`. Om filen saknas hämtas den automatiskt från projektets GitHub-repository. Internet behövs för hämtningen. Variabler finns kvar i minnet mellan cellerna, men försvinner när sessionen startas om. Sparade utskrifter i notebooken betyder inte att variablerna finns i den nya sessionen.

## Installera bibliotek

`%pip` är ett notebook-kommando som installerar ett bibliotek i den aktuella Python-miljön. Installation gör biblioteket tillgängligt; `import` i nästa cell laddar det så att koden kan använda det.

Den här cellen installerar både `requests` och `matplotlib`. `os`, `json`, `csv` och `gzip` ingår i Python och behöver inte installeras separat.

In [ ]:
%pip install requests matplotlib


## Importer

Här importeras biblioteken som resten av notebooken använder. `as plt` ger `matplotlib.pyplot` det kortare namnet `plt`.

Om ett bibliotek saknas visar Python ett fel och avbryter cellen vid den importen. Installera biblioteket i installationscellen och kör sedan importcellen igen. Texten om att alla bibliotek är importerade visas bara om samtliga importer lyckades.

In [ ]:
# os hanterar mappar och kontrollerar om lokala filer finns.
import os

# json läser och sparar filmdata i JSON-format.
import json

# csv läser och skriver tabeller, även IMDb:s tabulatorseparerade TSV-fil.
import csv

# gzip läser IMDb:s komprimerade betygsfil.
import gzip

# requests hämtar data från Wikidata och IMDb.
import requests

# matplotlib.pyplot skapar diagram. plt är ett kortare namn för modulen.
import matplotlib.pyplot as plt

# Den här raden nås bara om alla importer lyckades.
print("Alla bibliotek är importerade.")


## Förbered exempeldata

Den här cellen hämtar `data/1990.json` från GitHub om filen saknas. Därför behöver datafilen inte laddas upp manuellt i Colab. Om filen redan finns behålls den.

In [ ]:
# Hämtar exempeldata om notebooken körs utan zipfilens data-mapp.
os.makedirs("data", exist_ok=True)
filename = "data/1990.json"

# En befintlig fil behålls, inklusive redan matchade IMDb-betyg.
if not os.path.exists(filename):
    url = "https://raw.githubusercontent.com/cccarllll/Carl_H_filmer/main/data/1990.json"
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        example_data = response.json()
        # Kontrollera grundformatet innan svaret sparas som filmdata.
        if (not isinstance(example_data, dict)
                or not isinstance(example_data.get("movies"), list)
                or not isinstance(example_data.get("imdb_matched"), bool)):
            raise ValueError("Exempelfilen har fel format.")
        with open(filename, "w", encoding="utf-8") as file:
            json.dump(example_data, file, ensure_ascii=False, indent=4)
        print("Exempeldata hämtades från GitHub.")
    except (requests.RequestException, ValueError, OSError) as error:
        raise RuntimeError(
            "Exempeldata kunde inte hämtas. Kontrollera internetanslutningen "
            "eller ladda upp zipfilens data/1990.json och kör cellen igen."
        ) from error
else:
    print("Använder befintlig data/1990.json.")


## Klasser: Film och RatedFilm

En klass beskriver hur ett objekt byggs och vilka metoder det har. `Film` lagrar titel och år. Varje filmobjekt har sina egna attribut. `self` syftar på objektet som metoden arbetar med.

`RatedFilm(Film)` anger att RatedFilm ärver från Film. `super().__init__(title, year)` använder basklassens konstruktor för att sätta titel och år på samma objekt. Barnklassen lägger sedan till betyg och röster. Betyget omvandlas till `float` och rösterna till `int`, så att jämförelser och sortering görs med tal.

Testet längst ned skapar två exempelobjekt och visar attributen och metoderna. Ett RatedFilm-objekt räknas också som en Film, vilket visas med `isinstance`. Med 600 röster ska `has_enough_votes()` ge `True`, eftersom gränsen är 500.

In [ ]:
# Basklass för en film med information som alla filmer har.
# En klass är en mall. Varje objekt får sina egna värden för titel och år.
class Film:
    # __init__ körs när objektet skapas. self syftar på just det objektet.
    def __init__(self, title, year):
        # Sparar parametern title som ett attribut i objektet.
        self.title = title
        # Sparar parametern year som ett attribut i objektet.
        self.year = year

    # Returnerar filmens år och titel i samma format som används vid utskrift.
    def get_title_with_year(self):
        # f-strängen sätter in attributens värden i texten. return lämnar tillbaka texten.
        return f"{self.year} - {self.title}"


# Barnklass som ärver titel och år från Film.
# Lägger till IMDb-betyg och röster.
# Film inom parentes anger basklassen som RatedFilm ärver från.
class RatedFilm(Film):
    def __init__(self, title, year, imdb_rating, imdb_votes):
        # Anropar Films konstruktor för att spara titel och år i samma objekt.
        super().__init__(title, year)
        # float omvandlar exempelvis texten '7.5' till decimaltalet 7.5.
        self.imdb_rating = float(imdb_rating)
        # int omvandlar exempelvis texten '600' till heltalet 600.
        self.imdb_votes = int(imdb_votes)

    # Kontrollerar om filmen har tillräckligt många röster för topplistan.
    def has_enough_votes(self):
        # Jämförelsen ger True vid minst 500 röster, annars False.
        return self.imdb_votes >= 500


example_film = Film("Testfilm", 1990)
# Skapar ett objekt från barnklassen med extra attribut för betyg och röster.
example_rated_film = RatedFilm("Testfilm", 1990, 7.5, 600)



print("TEST AV BASKLASSEN FILM")
print("------------------------")
# type hittar objektets klass och __name__ ger klassens namn som text.
print(f"Objektets klass: {type(example_film).__name__}")
print(f"Titel: {example_film.title}")
print(f"År: {example_film.year}")
print(f"Formaterad titel: {example_film.get_title_with_year()}")

print("\nTEST AV BARNKLASSEN RATEDFILM")
print("-----------------------------")
print(f"Objektets klass: {type(example_rated_film).__name__}")
# isinstance kontrollerar om objektet är en Film, även genom arv.
print(f"Ärver från Film: {isinstance(example_rated_film, Film)}")
print(f"Titel: {example_rated_film.title}")
print(f"År: {example_rated_film.year}")
print(f"IMDb-betyg: {example_rated_film.imdb_rating}")
print(f"Antal IMDb-röster: {example_rated_film.imdb_votes}")
print(f"Formaterad titel: {example_rated_film.get_title_with_year()}")

print("\nTEST AV GRÄNSEN FÖR TOPPLISTAN")
print("------------------------------")
print("Minst 500 röster krävs för att filmen ska vara med.")
print(f"Filmens antal röster: {example_rated_film.imdb_votes}")
print(f"Har filmen tillräckligt många röster? "
      f"{example_rated_film.has_enough_votes()}")

print(
    "\nResultat: RatedFilm ärver titel, år och metoden "
    "get_title_with_year() från Film och lägger till "
    "IMDb-betyg och antal röster."
)


## Hämta filmdata från Wikidata

Funktionen tar emot ett årtionde, exempelvis 1990, och bygger en fråga för åren 1990–1999. Wikidata returnerar JSON med filmuppgifter. Här väljs filmer som har Sverige angivet som ursprungsland; det kan även omfatta samproduktioner och filmer på andra språk.

Rådata betyder här svaret innan min rensning. Varje träff innehåller nästlade dictionaries, till exempel `movie['filmLabel']['value']`. Samma film kan förekomma flera gånger med olika releasedatum.

När denna cell körs skapas data-mappen och funktionen definieras. Inga filmer hämtas av själva demonstrationstexten. Hämtningen sker senare om `main()` inte hittar lokal data. Utskriften att funktionen är laddad bekräftar inte att API:t har testats.

In [ ]:
# Skapar mappen data där programmets CSV- och JSON-filer sparas.
# exist_ok=True gör att programmet fortsätter även om mappen redan finns.
os.makedirs("data", exist_ok=True)


# Hämtar svenska filmer från Wikidata för hela det valda årtiondet.
# Frågan försöker även hämta originaltitel, originalspråk, regissör, genre,
# speltid, IMDb-id och beskrivning.
# IMDb-betyg och antal röster hämtas senare från IMDbs egen fil.
# OPTIONAL gör att en film tas med även när något av dessa extra fält saknas.
def get_swedish_movies_by_decade(decade):
    end_year = decade + 9

    query = f"""
    SELECT ?filmLabel ?releaseDate
           (GROUP_CONCAT(DISTINCT ?directorLabel; separator=", ") AS ?directors)
           (GROUP_CONCAT(DISTINCT ?genreLabel; separator=", ") AS ?genres)
           (SAMPLE(?duration) AS ?duration)
           (SAMPLE(?imdbId) AS ?imdbId)
           (SAMPLE(?description) AS ?description)
           (SAMPLE(?originalTitle) AS ?originalTitle)
           (GROUP_CONCAT(DISTINCT ?originalLanguageLabel; separator=", ") AS ?originalLanguages)
    WHERE {{
      ?film wdt:P31/wdt:P279* wd:Q11424;
            wdt:P495 wd:Q34;
            wdt:P577 ?releaseDate.
      FILTER(YEAR(?releaseDate) >= {decade} && YEAR(?releaseDate) <= {end_year})

      OPTIONAL {{
        ?film wdt:P57 ?director.
        ?director rdfs:label ?directorLabel.
        FILTER(LANG(?directorLabel) = "sv")
      }}
      OPTIONAL {{
        ?film wdt:P136 ?genre.
        ?genre rdfs:label ?genreLabel.
        FILTER(LANG(?genreLabel) = "sv")
      }}
      OPTIONAL {{ ?film wdt:P2047 ?duration. }}
      OPTIONAL {{ ?film wdt:P345 ?imdbId. }}
      OPTIONAL {{ ?film schema:description ?description. FILTER(LANG(?description) = "sv") }}
      OPTIONAL {{ ?film wdt:P1476 ?originalTitle. }}
      OPTIONAL {{
        ?film wdt:P364 ?originalLanguage.
        ?originalLanguage rdfs:label ?originalLanguageLabel.
        FILTER(LANG(?originalLanguageLabel) = "sv")
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "sv,en". }}
    }}
    GROUP BY ?filmLabel ?releaseDate
    ORDER BY ?releaseDate ?filmLabel
    """

    try:
        response = requests.get(
            "https://query.wikidata.org/sparql",
            params={"query": query, "format": "json"},
            headers={"User-Agent": "ExamenCarlH/0.2"},
            timeout=60,
        )
        # Visar serverns HTTP-status innan eventuella HTTP-fel hanteras.
        print(f"Wikidata svarade med statuskod: {response.status_code}")
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as error:
        print(f"Fel: Wikidata kunde inte svara just nu. Försök igen senare. {error}")
        raise SystemExit(1)

    return data["results"]["bindings"]


# Hämtningen görs av main när lokal filmdata saknas.
print("Wikidata-funktionen är laddad i minnet.")
print("Den används bara när lokal data saknas.")

## IMDb-data

IMDb:s komprimerade TSV-fil innehåller IMDb-id (`tconst`), betyg (`averageRating`) och antal röster (`numVotes`). TSV är tabelltext där kolumnerna skiljs åt med tabulatorer. Gzip komprimerar filen för att minska storleken.

`download_imdb_ratings()` kontrollerar först om `data/imdb_ratings.tsv.gz` finns. Om den finns återanvänds den. Annars laddas den ned och sparas. Funktionen returnerar filens sökväg, inte alla betyg.

`add_imdb_ratings()` kopplar ihop uppgifterna genom IMDb-id. En dictionary gör det möjligt att hitta mina filmer direkt med deras id medan betygsfilen läses rad för rad. Funktionen uppdaterar film-dictionaries i den befintliga listan. Filmer utan match behåller sina tidigare värden.

Utskrifterna med `callable` visar att funktionerna går att anropa. De laddar inte ned filen och testar inte att matchningen lyckas. När färdig lokal filmdata redan finns behöver huvudprogrammet inte läsa IMDb-filen igen.

In [ ]:

# Hämtar IMDbs officiella fil med betyg och antal röster till mappen data.
# Filen laddas bara ned om den saknas.
# IMDb-filen är gz-komprimerad och kommer senare läsas för att matcha våra IMDb-id.
# Om nedladdningen misslyckas skrivs ett fel ut och programmet avslutas.
def download_imdb_ratings():
    filename = "data/imdb_ratings.tsv.gz"

    # Kontrollerar om filen redan finns så att den kan återanvändas.
    if os.path.exists(filename):
        # Lämnar tillbaka sökvägen och avslutar funktionen.
        return filename

    url = "https://datasets.imdbws.com/title.ratings.tsv.gz"

    # Försöker köra blocket. Om ett passande fel uppstår går körningen till except.
    try:
        response = requests.get(url, timeout=60)
        # Visar serverns statuskod och kontrollerar sedan om svaret är ett HTTP-fel.
        print(f"IMDb svarade med statuskod: {response.status_code}")
        response.raise_for_status()

        # with stänger filen automatiskt. wb betyder skrivning av binära data.
        with open(filename, "wb") as file:
            # Sparar svarets råa bytes i filen, här den komprimerade IMDb-filen.
            file.write(response.content)
    # Fångar nätverksfel eller filfel och sparar felinformationen i error.
    except (requests.RequestException, OSError) as error:
        print(f"Fel: IMDb-filen kunde inte laddas ner. {error}")
        # Avslutar körningen med felkod 1. I Jupyter visas detta som ett undantag.
        raise SystemExit(1)

    # Lämnar tillbaka sökvägen och avslutar funktionen.
    return filename


# Matchar mina filmer mot IMDbs betygsfil med hjälp av varje films IMDb-id.
# Funktionen läser filen rad för rad och uppdaterar bara filmer som har ett IMDb-id.
# Returnerar True om minst ett betyg eller antal röster ändrades.
# Listan skickas in som argument. Film-dictionaries ändras direkt i den listan.
def add_imdb_ratings(movies, filename):
    # Bygger en dictionary där IMDb-id är nyckel och filmens dictionary är värde.
    movies_by_imdb_id = {
        # Tar bara med filmer med IMDb-id. Samma id kan ha ett värde i dictionaryn.
        movie["imdb_id"]: movie for movie in movies if movie.get("imdb_id")
    }
    # En boolesk variabel som håller reda på om något värde har ändrats.
    ratings_updated = False

    # Öppnar gzip-filen som läsbar text (rt). UTF-8 anger textens teckenkodning.
    with gzip.open(filename, "rt", encoding="utf-8") as file:
        # Varje rad blir en dictionary. \t betyder tabulator, som skiljer TSV-kolumnerna.
        reader = csv.DictReader(file, delimiter="\t")

        # Läser en rad i taget så att hela IMDb-tabellen inte behöver läggas i en lista.
        for row in reader:
            # tconst är IMDb-id. in kontrollerar om id:t finns bland dictionaryns nycklar.
            if row["tconst"] in movies_by_imdb_id:
                # Hämtar samma film-dictionary som ligger i movies. Ändringar syns därför där också.
                movie = movies_by_imdb_id[row["tconst"]]

                if movie.get("imdb_rating") != row["averageRating"]:
                    movie["imdb_rating"] = row["averageRating"]
                    # Markerar att minst ett värde har ändrats.
                    ratings_updated = True

                if movie.get("imdb_votes") != row["numVotes"]:
                    movie["imdb_votes"] = row["numVotes"]
                    # Markerar att minst ett värde har ändrats.
                    ratings_updated = True

    # Returnerar True om något ändrades, annars False.
    return ratings_updated

print("IMDb-funktioner laddade i minnet:")
# callable visar om namnet kan anropas. Funktionen körs inte av den här kontrollen.
print(f"download_imdb_ratings: {callable(download_imdb_ratings)}")
print(f"add_imdb_ratings: {callable(add_imdb_ratings)}")
print("\nIMDb-datafilen är inte läst ännu.")
print("Den läses först när movies har skapats efter rensningen.")

## Spara och läsa lokal data

Filmlistan sparas som både JSON och CSV. JSON behåller strukturen med en lista av dictionaries och används när programmet läser in filmer igen. CSV visar samma fält som en tabell med en film per rad.

`save_movies(movies, decade, imdb_matched)` skriver filerna, `load_movies(decade)` läser JSON-filen och `movie_data_exists(decade)` kontrollerar om sökvägen finns. Filnamnet byggs av årtiondet, exempelvis `data/1990.json`. Sökvägen är relativ till den mapp som Python körs från.

`with open(...)` ser till att filen stängs när blocket är färdigt. Läget `w` skriver över en befintlig fil. `json.load` läser från en fil och `json.dump` skriver till en fil.

Demonstrationen längst ned listar lokala JSON-filer utan att skriva över dem. Den här cellen kommer före rensningscellens demonstration, eftersom den demonstrationen använder `load_movies`. Kontrollen av sökvägen validerar inte innehållet. Vid inläsningen hanteras saknad fil, oläsbar JSON och felaktig grundstruktur med ett felmeddelande. Då avbryts körningen så att filen kan rättas.

JSON-filen innehåller nu ett objekt med `movies` (filmlistan) och `imdb_matched` (true eller false). `load_movies` returnerar båda värdena. False betyder att matchningen återstår; true betyder att den har körts, även om vissa filmer saknar betyg. Programmet läser bara det nya formatet. CSV innehåller fortfarande enbart filmtabellen.

`load_movies` läser `data/<årtionde>.json`. Den medföljande filen för 1990-talet innehåller Wikidata-information utan IMDb-betyg. `save_movies` uppdaterar filen lokalt när IMDb-matchningen är klar.

In [ ]:

# Sparar den rensade filmlistan i både CSV- och JSON-format.
# CSV kan öppnas i exempelvis Excel och JSON kan användas direkt av programmet.
# Filnamnet innehåller årtiondet så att olika sökningar inte blandas ihop.
def save_movies(movies, decade, imdb_matched):
    csv_filename = f"data/{decade}.csv"
    json_filename = f"data/{decade}.json"

    # w skapar eller skriver över filen. newline undviker extra tomrader i CSV.
    with open(csv_filename, "w", encoding="utf-8", newline="") as file:
        # Skriver dictionaries som tabellrader med kolumnerna som anges i fieldnames.
        writer = csv.DictWriter(
            file,
            # Namnen bestämmer både vilka fält CSV-filen får och deras kolumnordning.
            fieldnames=[
                "year",
                "title",
                "original_title",
                "wikidata_title",
                "original_languages",
                "directors",
                "genres",
                "duration",
                "imdb_id",
                "imdb_rating",
                "imdb_votes",
                "description",
                "information_level",
            ],
        )
        # Skriver kolumnnamnen på tabellens första rad.
        writer.writeheader()
        # Skriver en rad för varje film i listan.
        writer.writerows(movies)

    # Öppnar JSON-filen för skrivning. En befintlig fil med samma namn skrivs över.
    with open(json_filename, "w", encoding="utf-8") as file:
        # Sparar både filmlistan och om IMDb-matchningen har körts.
        # Python använder True/False; i JSON skrivs de som true/false.
        data = {"imdb_matched": imdb_matched, "movies": movies}
        json.dump(data, file, ensure_ascii=False, indent=4)

    print(f"\nFilmerna sparades i {csv_filename} och {json_filename}")


# Läser den sparade JSON-filen när data för årtiondet redan finns lokalt.
# Då behöver programmet inte hämta samma filmlista från Wikidata igen.
def load_movies(decade):
    filename = f"data/{decade}.json"


    # Läser JSON-objektet som innehåller markeringen och filmlistan.
    try:
        with open(filename, encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        print(f"Fel: filen {filename} saknas. Lägg den i data-mappen och försök igen.")
        raise SystemExit(1)
    except (json.JSONDecodeError, UnicodeError):
        print(f"Fel: {filename} innehåller inte läsbar JSON. Kontrollera filen.")
        raise SystemExit(1)
    except OSError as error:
        print(f"Fel: {filename} kunde inte läsas. {error}")
        raise SystemExit(1)

    # Accepterar bara vårt nya format med en boolesk markering och en lista.
    if (
        not isinstance(data, dict)
        or not isinstance(data.get("imdb_matched"), bool)
        or not isinstance(data.get("movies"), list)
    ):
        print(f"Fel: {filename} måste innehålla imdb_matched och movies i det nya formatet.")
        raise SystemExit(1)

    movies = data["movies"]
    # Kontrollerar att listan innehåller film-dictionaries med titel och år.
    if any(
        not isinstance(movie, dict) or not {"title", "year"}.issubset(movie)
        for movie in movies
    ):
        print(f"Fel: {filename} måste innehålla en filmlista med titel och år.")
        raise SystemExit(1)

    # Två returvärden tas emot som movies, imdb_matched i huvudprogrammet.
    return movies, data["imdb_matched"]


# Kontrollerar om en sparad JSON-fil redan finns för det valda årtiondet.
# Resultatet används för att avgöra om data ska läsas lokalt eller hämtas från Wikidata.
def movie_data_exists(decade):
    # Returnerar True om sökvägen finns, annars False. f-strängen sätter in årtiondet.
    return os.path.exists(f"data/{decade}.json")

# Skapar en tom lista som ska fyllas med filnamn.
local_json_files = []

# Går igenom namnen i data-mappen ett i taget.
for filename in os.listdir("data"):
    # Tar bara med namn som slutar med .json.
    if filename.endswith(".json"):
        # append lägger filnamnet sist i listan.
        local_json_files.append(filename)

# Sorterar listan på plats i namnordning.
local_json_files.sort()

print("Lokalt sparade JSON-filer:")

# En lista med innehåll räknas som True. En tom lista räknas som False.
if local_json_files:
    for filename in local_json_files:
        print(f"- {filename}")
# Körs när de föregående villkoren i samma if/elif-kedja inte stämde.
else:
    print("Inga lokala JSON-filer hittades.")

# Exempeldata används bara om årtiondets lokala arbetsfil saknas.
if os.path.exists("data/1990.json"):
    print("Medföljande Wikidata-exempel finns: data/1990.json")


## Rensa och visa filmdata

`clean_movies` tar emot råa Wikidata-träffar och returnerar en ny lista med enklare film-dictionaries. Varje dictionary har nycklar som `title`, `year` och `imdb_id`. Det är fortfarande Python-data i minnet; JSON är formatet som används när den sparas i en fil.

`cleaned_movies = []` skapar en lista som behåller filmernas ordning. `seen_titles = set()` skapar en mängd med unika titlar. Mängden används för att kontrollera om en titel redan har behandlats. `add` lägger till en titel, och `in` eller `not in` kontrollerar om den finns. `{}` utan innehåll skapar däremot en dictionary, inte en tom mängd.

Exempel: om träffarna heter A, A och B sparas den första A och sedan B. Detta är en enkel titelbaserad rensning: två olika filmer med samma titel kan slås ihop och extra information från senare träffar sammanfogas inte.

Informationsnivån anger hur många av fälten regissör, genre, speltid och IMDb-id som är ifyllda. Fullständig betyder att alla fyra finns; det betyder inte att uppgifterna är verifierade eller att IMDb-betyg finns.

Utskriftsfunktionerna visar filmer och räknar antal per år. Diagrammet använder samma årsräkning. Topplistan skapar RatedFilm-objekt, väljer filmer med minst 500 röster och sorterar efter betyg. Här är `movies` en lista av dictionaries, medan `rated_movies` är en lista av objekt.

Demonstrationen längst ned läser redan rensad `data/1990.json`, visar tre filmer, antal per år och topplistan. Först provas också själva rensningen med två små exempelträffar för samma titel. Det ska ge en film med grundläggande information. Den kräver den lokala filen, men gör inga nätanrop.

Med enbart exempeldata saknas betygen och topplistan är tom. Kör sista cellen med main() för att komplettera från IMDb, och kör sedan denna demonstrationscell igen om du vill se de sparade betygen.

In [ ]:

# Kontrollerar hur mycket av den viktigaste informationen som finns om en film.
# Fullständig betyder att regissör, genre, speltid och IMDb-id finns.
# Delvis betyder att minst ett av fälten finns och Grundläggande att de saknas.
# Parametern movie är en dictionary för en enda film.
def get_information_level(movie):
    # En lista med nycklarna till de fyra fält som används för informationsnivån.
    required_fields = ["directors", "genres", "duration", "imdb_id"]
    # bool blir False för tom text och True för innehåll. sum räknar True som 1.
    filled_fields = sum(bool(movie[field]) for field in required_fields)

    # len ger antalet fält. == jämför värden; = används för att tilldela ett värde.
    if filled_fields == len(required_fields):
        return "Fullständig"
    # Kontrolleras om föregående villkor var falskt. Här finns minst ett ifyllt fält.
    elif filled_fields > 0:
        return "Delvis"

    # Denna rad nås bara om inget av de tidigare villkoren redan har returnerat.
    return "Grundläggande"


# Tar bort filmer med samma titel och bygger en ren lista för resten av programmet.
# Originaltiteln används som titel när den finns och Wikidatas label sparas separat.
# Den rena listan har enkla fält som titel, år och genre i stället för rå JSON-data.
# Varje film får också en informationsnivå från get_information_level.
# Parametern movies är här rådata. Returvärdet får en annan, enklare struktur.
def clean_movies(movies):
    # En tom lista som ska innehålla de rensade filmerna.
    cleaned_movies = []
    # set() skapar en tom mängd med unika värden. Samma titel kan bara finnas en gång.
    # Mängden används för snabba medlemskontroller med in. Den har ingen bestämd ordning.
    seen_titles = set()

    # Går igenom listan. movie syftar på en films dictionary under varje varv.
    for movie in movies:
        # Läser en dictionary inuti en annan: först filmLabel, sedan dess value.
        title = movie["filmLabel"]["value"]

        # Tar bara med filmen om titeln ännu inte finns i mängden.
        if title not in seen_titles:
            # get ger ett standardvärde om nyckeln saknas. Här används {} och sedan tom text.
            original_title = movie.get("originalTitle", {}).get("value", "")

            # Skapar en dictionary med nyckel–värde-par för en film.
            cleaned_movie = {
                # or väljer originaltiteln om den är ifylld, annars används Wikidata-titeln.
                "title": original_title or title,
                # Nyckeln till vänster är fältnamnet; värdet till höger hämtas från variabeln.
                "original_title": original_title,
                "wikidata_title": title,
                # [:4] tar textens första fyra tecken, alltså årtalet i datumet.
                "year": movie["releaseDate"]["value"][:4],
                # Om directors saknas används en tom dictionary. Om value saknas används tom text.
                "directors": movie.get("directors", {}).get("value", ""),
                "genres": movie.get("genres", {}).get("value", ""),
                "duration": movie.get("duration", {}).get("value", ""),
                "imdb_id": movie.get("imdbId", {}).get("value", ""),
                "imdb_rating": movie.get("imdbRating", {}).get("value", ""),
                "imdb_votes": movie.get("imdbVotes", {}).get("value", ""),
                "description": movie.get("description", {}).get("value", ""),
                "original_languages": movie.get("originalLanguages", {}).get("value", ""),
            }
            # Anropar funktionen och sparar dess resultat under en ny nyckel.
            cleaned_movie["information_level"] = get_information_level(cleaned_movie)

            # Lägger den rensade filmen sist i listan.
            cleaned_movies.append(cleaned_movie)
            # add lägger titeln i mängden. Nästa träff med samma titel hoppas då över.
            # Två olika filmer med samma titel räknas också som dubletter med denna enkla metod.
            seen_titles.add(title)

    # Returnerar den nya listan efter att loopen gått igenom alla träffar.
    return cleaned_movies


# Skriver ut filmens årtal, titel, IMDb-betyg och antal IMDb-röster i terminalen.
# Saknas betyg eller röster i IMDb-filen visas texten Saknas i stället.
# Den här funktionen skriver ut data men har inget uttryckligt return; returvärdet blir None.
def print_movies(movies):
    # Går igenom listan. movie syftar på en films dictionary under varje varv.
    for movie in movies:
        title = movie["title"]
        # Hakparenteser läser värdet för en nyckel. Om nyckeln saknas uppstår KeyError.
        year = movie["year"]
        # get undviker fel om nyckeln saknas. or ger reservtext om värdet är tomt.
        rating = movie.get("imdb_rating") or "Saknas"
        # Samma reservtext används när antal röster saknas eller är tomt.
        votes = movie.get("imdb_votes") or "Saknas"

        print(f"{year} - {title} | IMDb: {rating} | Röster: {votes}")


# Räknar hur många filmer som finns för varje år i den aktuella filmlistan.
# Resultatet skrivs ut efter listan så att det blir lätt att se fördelningen per år.
def print_movie_count_by_year(movies):
    # {} skapar en tom dictionary. År blir nyckel och antal filmer blir värde.
    movies_per_year = {}

    # Går igenom listan. movie syftar på en films dictionary under varje varv.
    for movie in movies:
        # Hakparenteser läser värdet för en nyckel. Om nyckeln saknas uppstår KeyError.
        year = movie["year"]

        # Kontrollerar om året ännu saknas bland dictionaryns nycklar.
        if year not in movies_per_year:
            # Startar räknaren på noll första gången året förekommer.
            movies_per_year[year] = 0

        # Ökar årets räknare med ett. += 1 betyder lägg till 1 till det gamla värdet.
        movies_per_year[year] += 1

    print("\nAntal filmer per år:")

    # sorted ger en sorterad lista med dictionaryns nycklar, alltså åren.
    for year in sorted(movies_per_year):
        print(f"{year}: {movies_per_year[year]} filmer")

# Skapar ett stapeldiagram som visar hur många filmer som finns för varje år.
# Funktionen använder samma sparade filmlista som resten av programmet.
def show_movie_chart(movies, decade):
    # {} skapar en tom dictionary. År blir nyckel och antal filmer blir värde.
    movies_per_year = {}

    # Går igenom listan. movie syftar på en films dictionary under varje varv.
    for movie in movies:
        # Hakparenteser läser värdet för en nyckel. Om nyckeln saknas uppstår KeyError.
        year = movie["year"]

        # Kontrollerar om året ännu saknas bland dictionaryns nycklar.
        if year not in movies_per_year:
            # Startar räknaren på noll första gången året förekommer.
            movies_per_year[year] = 0

        # Ökar årets räknare med ett. += 1 betyder lägg till 1 till det gamla värdet.
        movies_per_year[year] += 1

    # Sorterar åren så att diagrammets staplar kommer i tidsordning.
    years = sorted(movies_per_year)
    # En list comprehension bygger antalslistan i samma ordning som åren.
    number_of_movies = [movies_per_year[year] for year in years]

    # Ritar en stapel per år. Listan med antal bestämmer staplarnas höjd.
    # En ny figur gör att upprepade menyval inte ritar ovanpå gamla staplar.
    plt.figure()
    plt.bar(years, number_of_movies)
    plt.title(f"Antal svenska filmer per år, {decade}-talet")
    # Sätter etiketten under diagrammets vågräta axel.
    plt.xlabel("År")
    # Sätter etiketten vid diagrammets lodräta axel.
    plt.ylabel("Antal filmer")
    # Visar det färdiga diagrammet.
    plt.show()

# Returnerar IMDb-betyget som används när RatedFilm-objekt ska sorteras.
def get_imdb_rating(movie):
    # Här är movie ett RatedFilm-objekt, så attributet läses med punktnotation.
    return movie.imdb_rating


# Sorterar filmer med IMDb-betyg från högst till lägst och visar de tio första.
# En film behöver minst 500 IMDb-röster för att vara med, så att ett fåtal röster
# inte får för stor påverkan på topplistan.
def print_top_rated_movies(movies):
    # En lista för RatedFilm-objekt som klarar topplistans krav.
    rated_movies = []

    # Går igenom listan. movie syftar på en films dictionary under varje varv.
    for movie in movies:
        # and kräver att både betyg och röster har ett värde.
        if movie.get("imdb_rating") and movie.get("imdb_votes"):
            # Skapar ett objekt från filmens dictionary. Konstruktorn omvandlar betyg och röster.
            rated_movie = RatedFilm(
                movie["title"],
                movie["year"],
                movie["imdb_rating"],
                movie["imdb_votes"],
            )

            # Anropar objektets metod som kontrollerar gränsen på 500 röster.
            if rated_movie.has_enough_votes():
                # Lägger till objektet i listan över godkända filmer.
                rated_movies.append(rated_movie)

    # key anger funktionen som hämtar sorteringsvärdet för varje objekt.
    # Inga parenteser efter funktionsnamnet: sort anropar den. reverse=True ger högst först.
    rated_movies.sort(key=get_imdb_rating, reverse=True)

    print("\nTopplista: högst IMDb-betyg (minst 500 röster)")

    # [:10] väljer högst tio filmer. enumerate ger både platsnummer och objekt.
    # start=1 gör att numreringen börjar på 1 i stället för 0.
    for position, movie in enumerate(rated_movies[:10], start=1):
        print(
            f"{position}. {movie.get_title_with_year()} | "
            f"IMDb: {movie.imdb_rating} | Röster: {movie.imdb_votes}"
        )


# Provar själva rensningen med två råa träffar för samma titel.
# Detta lilla exempel fungerar utan något API-anrop.
test_raw_movies = [
    {"filmLabel": {"value": "Testfilm"},
     "releaseDate": {"value": "1990-01-01T00:00:00Z"}},
    {"filmLabel": {"value": "Testfilm"},
     "releaseDate": {"value": "1991-01-01T00:00:00Z"}},
]
test_cleaned_movies = clean_movies(test_raw_movies)
print(f"Rensningstest: {len(test_raw_movies)} träffar blev "
      f"{len(test_cleaned_movies)} film.")
print(f"Informationsnivå: {test_cleaned_movies[0]['information_level']}")

# Läser redan rensad lokal data. Därför ska den inte skickas till clean_movies igen.
movies, imdb_matched = load_movies(1990)

print(f"IMDb-matchning gjord: {imdb_matched}")

print(f"Test med sparad 1990-data: {len(movies)} filmer lästes in.")

# \n börjar en ny rad före texten för att skilja utskriftsblocken åt.
print("\nTest av print_movies():")
# [:3] väljer de första tre filmerna för en kort demonstration.
print_movies(movies[:3])

print("\nTest av print_movie_count_by_year():")
print_movie_count_by_year(movies)

print("\nTest av print_top_rated_movies():")
print_top_rated_movies(movies)

## Meny

`show_menu(movies, decade)` får den färdiga filmlistan och visar de val användaren kan göra. En `while True`-loop visar menyn igen efter varje val. `break` avslutar loopen när användaren väljer 0.

`input` returnerar text, så valen jämförs med exempelvis `'1'`. `if` och `elif` väljer vilket funktionsanrop som ska köras. Ett ogiltigt val ger ett meddelande och ett nytt försök.

Cellen definierar menyn men öppnar den inte direkt. Menyn startas av `main()` när filmdata har laddats. Alla menyval använder samma lista i minnet.

In [ ]:
# Visar programmets enkla meny tills användaren väljer att avsluta.
# Val 1 visar filmlistan, val 2 räknar filmer per år och val 3 visar en topplista.
# Parametrarna tar emot listan och årtiondet från huvudprogrammet.
def show_menu(movies, decade):
    # Upprepar blocket tills break avslutar loopen eller return lämnar funktionen.
    while True:
        # print visar text för användaren; den lämnar inte tillbaka ett menyval.
        print("\nMeny")
        print("1. Visa alla filmer")
        print("2. Visa antal filmer per år")
        print("3. Visa topplista efter IMDb-betyg")
        print("4. Visa stapeldiagram över antal filmer per år")
        print("0. Avsluta")

        # input väntar på användaren och returnerar svaret som text.
        choice = input("Välj ett alternativ: ")

        if choice == "1":
            print(f"\n{decade}-talet")
            print_movies(movies)
        elif choice == "2":
            print_movie_count_by_year(movies)
        elif choice == "3":
            print_top_rated_movies(movies)
        elif choice == "4":
            show_movie_chart(movies, decade)
        elif choice == "0":
            print("Programmet avslutas.")
            # Avslutar den närmaste loopen.
            break
        # Körs när de föregående villkoren i samma if/elif-kedja inte stämde.
        else:
            print("Fel: välj 1, 2, 3, 4 eller 0.")

## Input: välj årtionde

Funktionen frågar efter ett årtionde mellan 1920 och 2020. Enter utan text ger standardvärdet 1990. `strip()` tar bort mellanslag runt svaret och `isdigit()` kontrollerar att texten består av siffror före omvandlingen med `int`.

`decade % 10 == 0` betyder att resten vid division med tio är noll. Därför accepteras 1990 men inte 1995. Funktionen avrundar alltså inte årtal till årtionden.

Vid ogiltig inmatning fortsätter loopen och frågar igen. `return` lämnar tillbaka det godkända heltalet och avslutar funktionen. När cellen bara definierar funktionen ställs ingen fråga ännu; frågan kommer när huvudprogrammet anropar den.

In [ ]:
# Frågar användaren efter ett giltigt årtionde mellan 1920 och 2020.
# Om användaren bara trycker Enter används 1990 som standard för att underlätta testning.
# Returnerar årtiondet direkt, till exempel 1990.
def get_decade_from_user():
    # Upprepar blocket tills break avslutar loopen eller return lämnar funktionen.
    while True:
        # input ger text. strip tar bort mellanslag i början och slutet.
        decade = input("Ange ett årtionde mellan 1920 och 2020 (Enter för 1990): ").strip()

        # Tom text betyder att användaren tryckte Enter utan att skriva ett värde.
        if decade == "":
            # Returnerar ett heltal som standardvärde; loopen fortsätter inte efter return.
            return 1990

        # Kontrollerar om texten består av siffror innan omvandlingen till heltal.
        if decade.isdigit():
            # Omvandlar texten till ett heltal så att den kan jämföras och användas i beräkningar.
            decade = int(decade)

            # Kontrollerar intervallet och att resten vid division med 10 är noll.
            # Därför godkänns 1990 men inte 1995.
            if 1920 <= decade <= 2020 and decade % 10 == 0:
                # Lämnar tillbaka det godkända årtiondet och avslutar funktionen.
                return decade

        print("Fel: ange ett årtionde mellan 1920 och 2020, till exempel 1990.")


## Main: programmets körordning

`main()` samlar stegen i den ordning de ska utföras. Först väljs årtiondet, sedan kontrolleras om JSON-filen redan finns.

Om filen finns läses filmerna och markeringen imdb_matched. Annars hämtas och rensas Wikidata-filmerna och sparas med false. Efter båda vägarna kontrolleras markeringen: bara vid false läses IMDb-filen och filmerna kompletteras. När matchningen lyckats sparas true. Till sist visas menyn.

Variabler som skapas inne i main, exempelvis `decade` och `movies`, är lokala för det funktionsanropet. De är separata från demonstrationernas variabler utanför funktionerna. Funktionerna delar data genom argument och returvärden.

Den här cellen definierar huvudfunktionen. Sista cellen anropar den och startar programmet.

In [ ]:
# Samlar programmets körning i en funktion så att flödet blir lätt att följa.
# Funktionskroppen körs först när main() anropas i sista cellen.
def main():
    # Hämtar årtiondet som användaren vill söka i.
    decade = get_decade_from_user()

    # Kontrollerar en gång om årtiondets data redan finns sparad lokalt.
    data_exists = movie_data_exists(decade)

    # Läser lokal arbetsdata eller medföljande exempeldata för årtiondet.
    if data_exists:
        print("Läser sparad filmdata...")
        movies, imdb_matched = load_movies(decade)

    # Hämtar och rensar filmer från Wikidata när data saknas lokalt.
    else:
        print("Hämtar filmdata från Wikidata. Det kan ta någon minut...")
        raw_movies = get_swedish_movies_by_decade(decade)

        print("Rensar och förbereder filmdata...")
        movies = clean_movies(raw_movies)

        # Nya Wikidata-filmer har ännu inte matchats mot IMDb.
        imdb_matched = False
        # Behåller de rensade filmerna även om IMDb-steget senare misslyckas.
        save_movies(movies, decade, imdb_matched)

    # Gäller både nya filmer och lokala filer som ännu inte matchats.
    if not imdb_matched:
        # Hämtar IMDbs betygsfil bara om den saknas lokalt.
        print("Läser IMDb-betyg och antal röster...")
        imdb_filename = download_imdb_ratings()

        # Fyller på den nya filmlistan med IMDb-betyg och antal röster.
        add_imdb_ratings(movies, imdb_filename)

        # Sparar den kompletta nya filmlistan med IMDb-data lokalt.
        # True betyder att matchningen är gjord, inte att varje film fått betyg.
        imdb_matched = True
        save_movies(movies, decade, imdb_matched)

    # Visar menyn när filmlistan är klar att användas.
    show_menu(movies, decade)



## Kör programmet

Kör alla föregående kodceller i ordning. Förberedelsecellen hämtar data/1990.json om den saknas, även i Colab. Tryck Enter för 1990. Programmet läser filmdata från data-mappen om filen finns. Om imdb_matched är false kompletteras filmerna från IMDb före menyn. Internet behövs om IMDb-filen saknas lokalt.

Välj 0 för att avsluta. Efter genomförd matchning sparas den kompletta listan i data/1990.json med true. Nästa körning behöver då inte läsa IMDb-filen igen. Vid omstart av Colabs runtime kan lokala filer behöva laddas upp igen.

In [ ]:
# Startar programmet
# Anropar huvudfunktionen. Alla funktioner den använder måste redan vara definierade.
main()